# 28_01 혼동행렬 기반 분류 평가

In [1]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

✅ 환경 설정 완료! 현재 적용된 폰트: ['Malgun Gothic']


# 01 정확도와 그 한계
모델 채점의 첫 단추 — 정확도의 정의, 그리고 불균형이 만드는 함정


### accuracy_score()로 정확도 구하기
정답을 먼저, 예측을 나중에 — 앞으로 모든 지표 함수가 따르는 인자 순서

In [2]:
# 코드
from sklearn import set_config

set_config(display="diagram")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [3]:
cdf = pd.read_csv("..\\Data\\28_cmapss_fd001_sample.csv", encoding="utf-8")

feats = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]

cX = cdf[feats]
cy = cdf["failure_soon"]

cX_train, cX_test, cy_train, cy_test = train_test_split(
    cX, cy, test_size=0.2, random_state=42, stratify=cy
)

c_model = RandomForestClassifier(n_estimators=100, random_state=42)
c_model.fit(cX_train, cy_train)

cy_pred = c_model.predict(cX_test)


from sklearn.metrics import accuracy_score
acc = accuracy_score(cy_test, cy_pred)
print("정확도:", acc)

정확도: 0.865


### 데이터 로드와 X·y 분리
CSV를 불러와 입력 X(센서 6종)와 정답 y(failure_soon)으로 분리


In [4]:
# 코드

print(cdf.head(10).to_markdown())
print()
print(cdf.tail(10).to_markdown())

feats = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]

cX = cdf[feats]
cy = cdf["failure_soon"]

|    |   sensor_2 |   sensor_3 |   sensor_4 |   sensor_7 |   sensor_11 |   sensor_15 |   RUL |   failure_soon |
|---:|-----------:|-----------:|-----------:|-----------:|------------:|------------:|------:|---------------:|
|  0 |    643.239 |    1598.76 |    1409.06 |    546.905 |      48.021 |       8.411 |    40 |              0 |
|  1 |    644.115 |    1589.06 |    1404.09 |    547.132 |      46.564 |       8.449 |    69 |              0 |
|  2 |    643.177 |    1593.63 |    1401.6  |    545.802 |      48.356 |       8.731 |     3 |              1 |
|  3 |    644.941 |    1593.51 |    1419.77 |    548.299 |      48.153 |       8.342 |    77 |              0 |
|  4 |    643.05  |    1614.13 |    1416.6  |    546.202 |      48.013 |       8.524 |    11 |              1 |
|  5 |    637.551 |    1589.7  |    1398.11 |    550.369 |      47.944 |       8.382 |   140 |              0 |
|  6 |    637.756 |    1586.33 |    1410.27 |    554.668 |      48.776 |       8.388 |   102 |          

### 분할·학습·예측
학습용·평가용으로 나눠 `RandomForest` 학습 후 예측 — 오늘 채점의 재료


In [5]:
# 코드
cX_train, cX_test, cy_train, cy_test = train_test_split(
    cX, cy, test_size=0.2, random_state=42, stratify=cy
)

c_model = RandomForestClassifier(n_estimators=100, random_state=42)
c_model.fit(cX_train, cy_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [6]:
cy_pred = c_model.predict(cX_test)

### 손계산과 함수 대조
같은 자리의 개수를 직접 세어 전체로 나눈 값과 함수 결과 비교

In [7]:
# 코드
correct = (cy_test == cy_pred).sum()
total = len(cy_test)

print("직접 계산:", correct / total)
print("함수 결과:", accuracy_score(cy_test, cy_pred))

직접 계산: 0.865
함수 결과: 0.865


### 더미 분류기로 기준선 만들기
`most_frequent` 전략 — 항상 가장 많은 클래스로만 예측하는 모델

In [8]:
# 코드
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(cX_train, cy_train)
print("베이스라인:", dummy.score(cX_test, cy_test))

베이스라인: 0.78


### 클래스 비율 확인
정상(0)과 고장 임박(1)의 개수·비율 확인 — 1이 얼마나 적은지 눈으로 확인


In [9]:
# 코드
print(cy.value_counts())
print(cy.value_counts(normalize=True))

failure_soon
0    781
1    219
Name: count, dtype: int64
failure_soon
0    0.781
1    0.219
Name: proportion, dtype: float64


### 무조건 정상 예측의 정확도
모두 정상으로 찍어도 정확도가 꽤 높음 — 함정을 눈으로 확인하는 순간


In [10]:
# 코드
all_normal = np.zeros(len(cy_test))
print(all_normal)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0.]


In [11]:
print("무조건 정상 데이터일 떄의 정확도는?:", accuracy_score(cy_pred, all_normal))

무조건 정상 데이터일 떄의 정확도는?: 0.825


### 두 모델 정확도 출력
더미와 모델의 정확도를 나란히 출력 — 두 숫자를 비교할 준비


In [12]:
# 코드
print("RandomForest 모델: ", accuracy_score(cy_pred, cy_test))
print("dummyClassifier 모델: ", accuracy_score(cy_pred, all_normal))

RandomForest 모델:  0.865
dummyClassifier 모델:  0.825


### 차이 계산과 판단
두 정확도의 차이를 퍼센트포인트로 — 1~2%p면 학습 거의 안 됨, 10%p 이상이면 의미 있음


In [13]:
# 코드
cmodel_acc = accuracy_score(cy_pred, cy_test)
dum_acc = accuracy_score(cy_pred, all_normal)

gap = cmodel_acc - dum_acc
print(f"두 모델 정확도 차이: {gap:.4f}")

두 모델 정확도 차이: 0.0400


# 02 혼동행렬
네 칸짜리 성적표 — TP·FN·FP·TN과 설비 비용의 언어


### confusion_matrix()와 출력 순서 주의
ravel로 펼치면 순서대로 tn, fp, fn, tp — 이 순서를 헷갈리면 모든 지표가 엉터리


In [14]:
# 코드

### 혼동행렬 시각화 (heatmap)
ConfusionMatrixDisplay로 색칠된 표 생성 — 진한 칸이 많은 칸


In [15]:
# 코드

### 예시로 네 칸 채우기
여섯 대를 한 칸씩 분류 → TP=2, FN=1, FP=1, TN=2 — 함수 결과와 일치 확인


In [16]:
# 코드

### 혼동행렬 생성
우리 모델 예측으로 혼동행렬 출력 — 2×2 배열 확인

In [17]:
# 코드

### heatmap 시각화
진한 칸이 어디인지 눈으로 확인 — FN 칸이 진하면 놓침 많다는 신호

In [18]:
# 코드

### 네 칸 추출
ravel로 네 칸을 각 변수에 담기 — 순서 tn, fp, fn, tp 엄수


In [19]:
# 코드

### 설비 대수로 해석 출력
각 칸을 현장 언어 문장으로 — 숫자가 현장의 의미로 바뀌는 순간


In [20]:
# 코드

# 03 정밀도와 재현율
경보의 신뢰도와 놓치지 않는 능력 — 두 지표의 짝과 트레이드오프


### 네 칸으로 손계산
공식에 네 칸을 직접 대입 — 정밀도 = TP/(TP+FP), 재현율 = TP/(TP+FN)


In [21]:
# 코드

### 함수와 대조
손계산과 함수 결과 비교 — 일치하면 공식을 완전히 이해


In [22]:
# 코드

### 세 지표 한 화면 출력
정확도·정밀도·재현율을 함께 출력 — 정확도 높은데 재현율 낮은 식의 치우침 확인


In [23]:
# 코드